# Heinzy - shared Gemma 2 12B host (Google Colab)

**Issue:** [#16](https://github.com/MahikaGunjkar/AIGovernance/issues/16)

This notebook is the **team Gemma host**. It runs Ollama with gemma2:12b on a Colab GPU and exposes Ollama's HTTP API through an **ngrok** tunnel so teammates can set:

`	ext
MODEL_ENDPOINT=https://<ngrok-host>
`

Heinzy's generator already calls {MODEL_ENDPOINT}/api/chat - no app code changes.

## Operator (@asriram15)

1. Runtime -> Change runtime type -> **GPU** (T4 or better).
2. Colab Secrets (key icon) -> add NGROK_AUTHTOKEN (from https://dashboard.ngrok.com/get-started/your-authtoken).
3. Run all cells.
4. Copy the printed MODEL_ENDPOINT=... line into the **team chat** and into local gitignored TEAM_INFRA_NOTES.md.
5. When Colab disconnects or you restart: run again and **post the new URL** - old endpoints go stale silently.

## Limits

- Colab sessions die after idle / max lifetime; the tunnel URL **changes every restart**.
- The tunnel is on the public internet. Prefer ngrok account protections; do not paste secrets into cells.
- Teammates: if generation stops working, ping **@asriram15** and re-check /api/tags.


In [ ]:
# 1) Confirm GPU runtime
import subprocess, sys
print(subprocess.getoutput('nvidia-smi')[:800] or 'WARNING: no GPU visible — Runtime → GPU')


In [ ]:
# 2) Install Ollama
import os, subprocess, time, urllib.request

OLLAMA_HOST = '127.0.0.1:11434'
os.environ['OLLAMA_HOST'] = OLLAMA_HOST

if not os.path.exists('/usr/local/bin/ollama'):
    subprocess.check_call('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
print('ollama:', subprocess.getoutput('ollama --version'))


In [ ]:
# 3) Start Ollama server (background)
import subprocess, time, urllib.request, os

OLLAMA_HOST = os.environ.get('OLLAMA_HOST', '127.0.0.1:11434')

def ollama_up() -> bool:
    try:
        urllib.request.urlopen(f'http://{OLLAMA_HOST}/api/tags', timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    # Serve on all interfaces so ngrok can reach it.
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for i in range(30):
        if ollama_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError('Ollama failed to start')
print('Ollama is up at', OLLAMA_HOST)


In [ ]:
# 4) Pull gemma2:12b (large — expect several minutes)
import subprocess
print(subprocess.getoutput('ollama pull gemma2:12b'))
print('--- tags ---')
print(subprocess.getoutput('ollama list'))


In [ ]:
# 5) Start ngrok tunnel → public MODEL_ENDPOINT
# Requires Colab Secret: NGROK_AUTHTOKEN
import os, time, urllib.request, json, subprocess, sys

try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception as e:
    raise RuntimeError('Run this on Google Colab with Secret NGROK_AUTHTOKEN') from e

if not token:
    raise RuntimeError('Add Colab Secret NGROK_AUTHTOKEN')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(token)
tunnel = ngrok.connect(11434, 'http')
public_url = tunnel.public_url.rstrip('/')
# Heinzy Generator expects a base URL with no trailing path.
model_endpoint = public_url
print('MODEL_ENDPOINT=' + model_endpoint)
print()
print('Paste into teammate .env (and team chat):')
print(f'MODEL_ENDPOINT={model_endpoint}')


In [ ]:
# 6) Self-check: /api/tags and a tiny /api/chat ping
import json, urllib.request

assert 'model_endpoint' in globals(), 'Run the ngrok cell first'

def get(path):
    with urllib.request.urlopen(model_endpoint + path, timeout=60) as r:
        return json.load(r)

tags = get('/api/tags')
names = [m.get('name') for m in tags.get('models', [])]
print('models:', names)
assert any('gemma2:12b' in (n or '') for n in names), 'gemma2:12b not in /api/tags — re-run pull cell'

req = urllib.request.Request(
    model_endpoint + '/api/chat',
    data=json.dumps({
        'model': 'gemma2:12b',
        'messages': [{'role': 'user', 'content': 'Reply with exactly: pong'}],
        'stream': False,
    }).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=180) as r:
    data = json.load(r)
print('chat ok:', data.get('message', {}).get('content', '')[:200])
print()
print('READY — keep this runtime awake; post MODEL_ENDPOINT to the team.')
